# Quick demo: context-aware English↔Chinese chat translation (TowerInstruct-7B)

About **10–15 minutes** in total, most of it downloading the model once.

1. A small check on 3 real BMELD conversations (both directions, with and without context, scored against references).
2. An input box: type chat messages (`en: ...` or `zh: ...`) and get translations that use the conversation so far as context.

**Settings first:** Settings → Accelerator → **GPU T4 x2**, and Internet **On**.

In [ ]:
# 1. GPUs (should list two Tesla T4)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Get the code + a small part of the data (private repo: add a Kaggle secret GITHUB_TOKEN first)
import os, subprocess
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass
url = f"https://{token+'@' if token else ''}github.com/namannamu18/context-aware-translation.git"
os.chdir("/kaggle/working")
if not os.path.exists("cat"):
    subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", url, "cat"], check=True)
os.chdir("/kaggle/working/cat")
subprocess.run(["git", "sparse-checkout", "set", "scripts", "raw_data/mt/bmeld_test.zh", "raw_data/mt/synthetic_chat_test.zh"], check=True)
!pip install -q sacrebleu
print("ready")

In [ ]:
# 3. Load TowerInstruct-7B once (float16, split over both T4s). First time: ~5-10 min download.
import os, sys
os.environ["USE_TF"] = "0"; os.environ["TRANSFORMERS_NO_TF"] = "1"  # skip Kaggle's preinstalled TensorFlow
sys.path.insert(0, "/kaggle/working/cat/scripts")
from translate_chat import ChatTranslator, quick_check, interactive
tr = ChatTranslator("Unbabel/TowerInstruct-7B-v0.2", dtype="float16", device_map="auto")
print("model loaded")

In [ ]:
# 4. Small check: 3 real BMELD test conversations, both directions, with and without context (~2-4 min)
from pathlib import Path
results = quick_check(tr, Path("/kaggle/working/cat"), "bmeld_test", n_docs=3, n_show=8)

## Try it yourself

Type one chat message per line, starting with its language:

- `en: Hi, I ordered a coat last week but it is too small.`
- `zh: 那我可以换一件大一点的吗？`

Each message is translated into the other language **using the earlier messages as context**
(for comparison, the no-context translation is shown too). `reset` starts a new conversation, `quit` stops.

In [ ]:
# 5. Interactive translation
interactive(tr)

In [ ]:
# 6. Or call it directly from code
history = [("en", "Hi, I bought a coat last week."), ("zh", "尺码太小了。")]
print(tr.translate("Would you like to exchange it for a larger one?", "en", "zh", history))
print(tr.translate("Would you like to exchange it for a larger one?", "en", "zh", None, use_context=False))